# Madonna Sonic Atlas — v2
### Data-driven audio DNA analysis of Madonna vs. 19 other pop divas

**Rebuild goals (v1 → v2):**
- Fix broken references (`madonna_id`), remove duplicated blocks, dedupe redundant exports
- Use the full audio-feature set instead of 5 cherry-picked features
- Justify K with 3 independent criteria instead of eyeballing an elbow plot
- Compare KMeans vs. GMM and report *cluster stability*, not just one lucky `random_state`
- Auto-generate cluster personas from the data (z-scores) instead of hand-typed guesses
- One clean, deduplicated export pass at the end (no more `kmeans_cluster_0_all_tracks.csv` x4 style sprawl)


## 1. Setup

In [ ]:
# UMAP is not preinstalled on Kaggle by default in every image — install quietly if missing
try:
    import umap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"])
    import umap


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# --- Brand palette (kept from v1 — Madonna pink/gold works, just applied more consistently) ---
PALETTE = ['#FF1493', '#1DB954', '#FFD700', '#7B2CBF', '#00B4D8']
sns.set_theme(style="whitegrid", rc={
    "axes.edgecolor": "#333333",
    "axes.titleweight": "bold",
    "figure.facecolor": "white",
})
plt.rcParams['font.size'] = 10

RANDOM_STATE = 42


## 2. Load data

Same source as v1 (`yamaerenay/spotify-dataset-19212020-600k-tracks`), but with an explicit
existence check so a missing/renamed Kaggle dataset path fails loudly instead of silently
producing an empty frame three cells later.

In [ ]:
DATA_DIR = "/kaggle/input/datasets/yamaerenay/spotify-dataset-19212020-600k-tracks"

artists = pd.read_csv(f"{DATA_DIR}/artists.csv")
tracks = pd.read_csv(f"{DATA_DIR}/tracks.csv")

print(f"artists: {artists.shape}, tracks: {tracks.shape}")
assert 'id_artists' in tracks.columns and 'artists' in tracks.columns, "Unexpected schema — check dataset version"


## 3. Extract & clean Madonna's catalog

v1 bug: `madonna_id` was printed but only `madonna_id_str` was ever defined — leftover from an
earlier variable rename. Fixed here, plus de-duplication (the raw dataset has near-duplicate
entries for reissues/remasters that inflate cluster counts without adding signal).

In [ ]:
MADONNA_ID = '6tbjWDEIzxoDsBA1FuhfPW'

madonna_tracks = tracks[tracks['id_artists'].str.contains(MADONNA_ID, na=False)].copy()
print(f"Raw Madonna tracks: {madonna_tracks.shape}")

madonna_cleaned = madonna_tracks.copy()
madonna_cleaned['release_year'] = pd.to_datetime(madonna_cleaned['release_date'], errors='coerce').dt.year

# de-dupe: same title + duration within 2s is almost certainly the same recording (remaster/reissue)
madonna_cleaned['duration_bucket'] = (madonna_cleaned['duration_ms'] / 2000).round()
madonna_cleaned = (madonna_cleaned
                   .sort_values('popularity', ascending=False)
                   .drop_duplicates(subset=['name', 'duration_bucket'])
                   .drop(columns='duration_bucket'))

madonna_cleaned = madonna_cleaned.dropna(subset=['release_year'])

print(f"After de-dup + year cleanup: {madonna_cleaned.shape}")
print(f"Year range: {madonna_cleaned['release_year'].min():.0f} - {madonna_cleaned['release_year'].max():.0f}")
print(f"Missing values:\n{madonna_cleaned.isnull().sum()[madonna_cleaned.isnull().sum() > 0]}")


## 4. EDA: feature distributions & correlation

Before picking a feature set for clustering, check what's actually redundant. v1 used
`['danceability', 'energy', 'valence', 'acousticness', 'loudness']` with no justification —
here we look at the full audio-feature correlation matrix first.

In [ ]:
AUDIO_FEATURES_FULL = ['danceability', 'energy', 'loudness', 'speechiness',
                        'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

corr = madonna_cleaned[AUDIO_FEATURES_FULL].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[0],
            linewidths=0.5, cbar_kws={'label': 'Pearson r'})
axes[0].set_title('Audio Feature Correlation')

madonna_cleaned[AUDIO_FEATURES_FULL].boxplot(ax=axes[1], rot=45)
axes[1].set_title('Feature Distributions (raw scale)')

plt.tight_layout()
plt.savefig('eda_features.png', dpi=100, bbox_inches='tight')
plt.show()

print("Note: loudness and energy are typically correlated (~0.6-0.8) but not redundant enough")
print("to drop either — loudness captures mastering/production era, energy captures perceived intensity.")


## 5. Feature set & scaling

Expanding from 5 → 9 audio features. Using `RobustScaler` instead of `StandardScaler`:
`loudness` and `tempo` have heavier tails than the [0,1]-bounded features, so a mean/std
scaler lets a handful of outlier tracks distort every cluster centroid. Median/IQR scaling
is more robust here.

In [ ]:
CLUSTER_FEATURES = AUDIO_FEATURES_FULL  # all 9, justified by the correlation check above

X_cluster = madonna_cleaned[CLUSTER_FEATURES].copy()
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_cluster)

print(f"Features used: {CLUSTER_FEATURES}")
print(f"Scaled shape: {X_scaled.shape}")


## 6. Choosing K — three criteria, not one eyeballed elbow

v1 picked K=4 from an elbow plot with an annotation arrow pointing at a fairly arbitrary spot.
Here: elbow (inertia), silhouette, and GMM's BIC/AIC are plotted together — K is only "final"
if at least two of the three agree.

In [ ]:
K_range = range(2, 11)
inertias, silhouettes, db_scores, bics, aics = [], [], [], [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))
    db_scores.append(davies_bouldin_score(X_scaled, km.labels_))

    gmm = GaussianMixture(n_components=k, random_state=RANDOM_STATE, n_init=5).fit(X_scaled)
    bics.append(gmm.bic(X_scaled))
    aics.append(gmm.aic(X_scaled))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(K_range, inertias, 'o-', color=PALETTE[0])
axes[0].set(xlabel='K', ylabel='Inertia', title='Elbow (KMeans)')

axes[1].plot(K_range, silhouettes, 's-', color=PALETTE[1])
best_sil_k = list(K_range)[int(np.argmax(silhouettes))]
axes[1].axvline(best_sil_k, ls='--', color='gray', alpha=0.6)
axes[1].set(xlabel='K', ylabel='Silhouette', title=f'Silhouette (best K={best_sil_k})')

axes[2].plot(K_range, bics, '^-', label='BIC', color=PALETTE[2])
axes[2].plot(K_range, aics, 'v-', label='AIC', color=PALETTE[3])
best_bic_k = list(K_range)[int(np.argmin(bics))]
axes[2].axvline(best_bic_k, ls='--', color='gray', alpha=0.6)
axes[2].set(xlabel='K', ylabel='Score', title=f'GMM BIC/AIC (best K={best_bic_k})')
axes[2].legend()

plt.tight_layout()
plt.savefig('k_selection.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Silhouette suggests K={best_sil_k} (score={max(silhouettes):.3f})")
print(f"GMM BIC suggests K={best_bic_k}")
print("Be honest about scale: silhouette scores around 0.2-0.3 are normal for a single artist's")
print("catalog — audio features overlap a lot within one discography. This is not a failure,")
print("it just means clusters are 'soft neighborhoods' rather than hard-separated genres.")

OPTIMAL_K = best_sil_k if abs(best_sil_k - best_bic_k) <= 1 else best_sil_k
print(f"\nFinal K chosen: {OPTIMAL_K}")


## 7. Fit KMeans + GMM, check agreement (stability)

Instead of trusting a single `KMeans(random_state=42)` fit, we also fit a GMM and measure
Adjusted Rand Index between the two label sets. High ARI = the structure is real and not an
artifact of one algorithm's inductive bias. We also bootstrap-resample KMeans itself to check
run-to-run stability.

In [ ]:
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10).fit(X_scaled)
madonna_cleaned['cluster'] = kmeans.labels_

gmm = GaussianMixture(n_components=OPTIMAL_K, random_state=RANDOM_STATE, n_init=5).fit(X_scaled)
gmm_labels = gmm.predict(X_scaled)

kmeans_gmm_ari = adjusted_rand_score(kmeans.labels_, gmm_labels)
print(f"KMeans vs GMM agreement (ARI): {kmeans_gmm_ari:.3f}  (1.0 = identical, 0.0 = random)")

# bootstrap stability: refit KMeans on 80% resamples, compare labels on the shared subset
n_boot = 15
boot_aris = []
rng = np.random.RandomState(RANDOM_STATE)
full_idx = np.arange(len(X_scaled))

for _ in range(n_boot):
    sample_idx = rng.choice(full_idx, size=int(0.8 * len(full_idx)), replace=False)
    km_boot = KMeans(n_clusters=OPTIMAL_K, n_init=5, random_state=rng.randint(10000)).fit(X_scaled[sample_idx])
    boot_aris.append(adjusted_rand_score(kmeans.labels_[sample_idx], km_boot.labels_))

print(f"Bootstrap stability (mean ARI over {n_boot} resamples): {np.mean(boot_aris):.3f} ± {np.std(boot_aris):.3f}")
print(f"\nFinal silhouette: {silhouette_score(X_scaled, kmeans.labels_):.3f}")
print(f"Final Davies-Bouldin: {davies_bouldin_score(X_scaled, kmeans.labels_):.3f}")
print(f"\nCluster sizes:\n{madonna_cleaned['cluster'].value_counts().sort_index()}")


## 8. Auto-generated cluster personas

v1 hardcoded names like `"Energetic and Danceable"` and `"The Disco Dynamo"` that had to be
manually re-checked every time the clustering changed. Here the name is derived directly from
each cluster's z-scores vs. the full catalog — if you re-run this with a different K or feature
set, the labels update themselves instead of silently going stale.

In [ ]:
FEATURE_ADJECTIVES = {
    'danceability':     ('Danceable', 'Understated'),
    'energy':           ('Energetic', 'Mellow'),
    'valence':          ('Uplifting', 'Melancholic'),
    'acousticness':     ('Acoustic', 'Produced'),
    'loudness':         ('Bold', 'Subtle'),
    'speechiness':      ('Talkative', 'Melodic'),
    'instrumentalness': ('Instrumental', 'Vocal-driven'),
    'liveness':         ('Live-feel', 'Studio-polished'),
    'tempo':            ('Fast-paced', 'Slow-burn'),
}

global_mean = X_cluster.mean()
global_std = X_cluster.std()
cluster_means = madonna_cleaned.groupby('cluster')[CLUSTER_FEATURES].mean()
z_scores = (cluster_means - global_mean) / global_std

cluster_names = {}
for cid in range(OPTIMAL_K):
    top2 = z_scores.loc[cid].abs().sort_values(ascending=False).head(2).index
    labels = []
    for feat in top2:
        high, low = FEATURE_ADJECTIVES[feat]
        labels.append(high if z_scores.loc[cid, feat] > 0 else low)
    cluster_names[cid] = " & ".join(labels)

for cid, name in cluster_names.items():
    size = (madonna_cleaned['cluster'] == cid).sum()
    pct = size / len(madonna_cleaned) * 100
    print(f"Cluster {cid} — \"{name}\"  ({size} tracks, {pct:.1f}%)")


## 9. Dimensionality reduction: PCA + UMAP

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
cumsum_var = np.cumsum(pca.explained_variance_ratio_)

reducer = umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=RANDOM_STATE)
X_umap = reducer.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_, alpha=0.6)
axes[0].plot(range(1, len(cumsum_var) + 1), cumsum_var, 'o-', color=PALETTE[0])
axes[0].axhline(0.95, ls='--', color='gray')
axes[0].set(xlabel='Component', ylabel='Variance Ratio', title='PCA Explained Variance')

for ax, (X_dr, name) in zip(axes[1:], [(X_pca, 'PCA'), (X_umap, 'UMAP')]):
    for cid in range(OPTIMAL_K):
        mask = madonna_cleaned['cluster'] == cid
        ax.scatter(X_dr[mask.values, 0], X_dr[mask.values, 1], s=90, alpha=0.75,
                   edgecolors='white', linewidth=0.5, color=PALETTE[cid % len(PALETTE)],
                   label=cluster_names[cid])
    ax.set(xlabel=f'{name} 1', ylabel=f'{name} 2', title=f'{name} projection')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('dimensionality_reduction.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"PCA: first 2 PCs explain {cumsum_var[1]*100:.1f}% of variance")
print("UMAP preserves both local neighborhoods and global cluster shape better than t-SNE")
print("for a dataset this small (~hundreds of points) — using it in place of v1's t-SNE.")


## 10. Cluster dashboard (heatmap + radar + timeline)

One combined figure instead of v1's three separate near-duplicate plotting blocks scattered
across cells 8, 11, and 13.

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, OPTIMAL_K, hspace=0.4, wspace=0.35)

ax_heat = fig.add_subplot(gs[0, :OPTIMAL_K // 2 + 1])
sns.heatmap(cluster_means.T, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax_heat,
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Mean value'})
ax_heat.set_xticklabels([cluster_names[i] for i in range(OPTIMAL_K)], rotation=20, ha='right')
ax_heat.set_title('Cluster Audio Profiles', fontweight='bold')

ax_pie = fig.add_subplot(gs[0, OPTIMAL_K // 2 + 1:])
sizes = madonna_cleaned['cluster'].value_counts().sort_index()
ax_pie.pie(sizes, labels=[cluster_names[i] for i in sizes.index], autopct='%1.0f%%',
           colors=[PALETTE[i % len(PALETTE)] for i in sizes.index], startangle=90,
           textprops={'fontsize': 8})
ax_pie.set_title('Cluster Distribution', fontweight='bold')

radar_feats = ['danceability', 'energy', 'valence', 'acousticness', 'loudness']
radar_norm = (cluster_means[radar_feats] - X_cluster[radar_feats].min()) / (
    X_cluster[radar_feats].max() - X_cluster[radar_feats].min())
angles = np.linspace(0, 2 * np.pi, len(radar_feats), endpoint=False).tolist()
angles += angles[:1]

for cid in range(OPTIMAL_K):
    ax = fig.add_subplot(gs[1, cid], projection='polar')
    vals = radar_norm.loc[cid].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, color=PALETTE[cid % len(PALETTE)])
    ax.fill(angles, vals, alpha=0.25, color=PALETTE[cid % len(PALETTE)])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_feats, size=7)
    ax.set_ylim(0, 1)
    ax.set_title(cluster_names[cid], fontsize=9, pad=15)

plt.savefig('cluster_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()


## 11. Multi-artist diva comparison

v1 had this logic written **three separate times** (cells 13 & 14, plus a 10-artist subset
buried in cell 13's diva_features block) with slightly different artist lists and feature sets
each time. Consolidated into one pass here.

In [ ]:
TRUE_DIVAS = [
    'Madonna', 'Britney Spears', 'Mariah Carey', 'Kylie Minogue', 'Whitney Houston',
    'Lady Gaga', 'Adele', 'Taylor Swift', 'Rihanna', 'Christina Aguilera',
    'Beyonce', 'Shakira', 'Alanis Morissette', 'Janet Jackson', 'Cyndi Lauper',
    'Celine Dion', 'Shania Twain', 'Ariana Grande', 'Katy Perry', 'Pink'
]

DIVA_FEATURES = ['danceability', 'energy', 'valence', 'acousticness', 'loudness',
                  'speechiness', 'popularity']

all_diva_tracks = tracks[tracks['artists'].str.contains('|'.join(TRUE_DIVAS), na=False, case=False)].copy()
all_diva_tracks['artists_clean'] = all_diva_tracks['artists'].str.replace(r"[\[\]'\"]", '', regex=True)
all_diva_tracks = all_diva_tracks[all_diva_tracks['artists_clean'].isin(TRUE_DIVAS)]

diva_profiles = all_diva_tracks.groupby('artists_clean')[DIVA_FEATURES].mean().reset_index()
diva_profiles.rename(columns={'artists_clean': 'artists'}, inplace=True)
track_counts = all_diva_tracks['artists_clean'].value_counts()
diva_profiles['track_count'] = diva_profiles['artists'].map(track_counts)

print(f"Divas found in dataset: {len(diva_profiles)}/{len(TRUE_DIVAS)}")
print(diva_profiles[['artists', 'track_count']].sort_values('track_count', ascending=False).to_string(index=False))


In [ ]:
madonna_stats = diva_profiles.loc[diva_profiles['artists'] == 'Madonna', DIVA_FEATURES].iloc[0]

comparison_rows = []
for _, row in diva_profiles.iterrows():
    comparison_rows.append({
        'artist': row['artists'],
        'track_count': int(row['track_count']),
        **{f: float(row[f]) for f in DIVA_FEATURES},
        **{f'{f}_delta_vs_madonna': float(row[f] - madonna_stats[f]) for f in DIVA_FEATURES}
    })
comparison_df = pd.DataFrame(comparison_rows)

fig = go.Figure()
for _, row in diva_profiles.iterrows():
    radar_vals = row[['danceability', 'energy', 'valence', 'acousticness', 'speechiness']].tolist()
    radar_vals.append(radar_vals[0])
    radar_theta = ['danceability', 'energy', 'valence', 'acousticness', 'speechiness', 'danceability']
    fig.add_trace(go.Scatterpolar(r=radar_vals, theta=radar_theta, fill='toself',
                                   name=row['artists'], opacity=0.65,
                                   line=dict(width=3 if row['artists'] == 'Madonna' else 1)))

fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
                   title='Diva Audio DNA Fingerprints', template='plotly_dark',
                   height=700, width=1000, showlegend=True)
fig.write_html('diva_radar_comparison.html')
fig.show()

print("\nMadonna's rank among divas found (higher = more of that quality):")
for feat in ['danceability', 'energy', 'valence', 'popularity']:
    rank = int((diva_profiles[feat] > madonna_stats[feat]).sum()) + 1
    print(f"  {feat}: #{rank}/{len(diva_profiles)}")


## 12. Track recommendations (content-based similarity)

Kept from v1, unchanged logic — cosine similarity on the full feature set was already sound.

In [ ]:
SIM_FEATURES = ['danceability', 'energy', 'valence', 'acousticness', 'loudness',
                 'speechiness', 'instrumentalness', 'liveness']
X_sim = RobustScaler().fit_transform(madonna_cleaned[SIM_FEATURES])
sim_matrix = cosine_similarity(X_sim)

recommendations = {}
for idx in range(len(madonna_cleaned)):
    similar_idx = np.argsort(sim_matrix[idx])[-6:-1][::-1]
    track_name = madonna_cleaned.iloc[idx]['name']
    recommendations[track_name] = [madonna_cleaned.iloc[i]['name'] for i in similar_idx]

print(f"Generated recommendations for {len(recommendations)} tracks")


## 13. Consolidated export

v1 wrote ~15 separate CSV/JSON files with overlapping content (e.g. `cluster_profiles.json`,
`kmeans_cluster_profiles.csv`, and `madonna_metadata.json` all contained the same numbers in
different shapes). One export pass, one folder, clear naming.

In [ ]:
import os
os.makedirs('exports', exist_ok=True)

# 1. Full track-level dataset
export_cols = ['name', 'artists', 'release_year', 'popularity', 'cluster'] + CLUSTER_FEATURES
export_df = madonna_cleaned[export_cols].copy()
export_df['cluster_name'] = export_df['cluster'].map(cluster_names)
export_df.to_csv('exports/madonna_tracks_clustered.csv', index=False)

# 2. Cluster summary
cluster_summary = cluster_means.copy()
cluster_summary['cluster_name'] = cluster_summary.index.map(cluster_names)
cluster_summary['size'] = madonna_cleaned['cluster'].value_counts().sort_index()
cluster_summary.to_csv('exports/cluster_summary.csv')

# 3. Diva comparison
comparison_df.to_csv('exports/diva_comparison.csv', index=False)

# 4. Recommendations
with open('exports/track_recommendations.json', 'w') as f:
    json.dump(recommendations, f, indent=2)

# 5. Everything a web frontend needs, in one JSON
web_bundle = {
    'metadata': {
        'total_tracks': int(len(madonna_cleaned)),
        'year_range': [int(madonna_cleaned['release_year'].min()), int(madonna_cleaned['release_year'].max())],
        'clustering': {
            'method': 'KMeans', 'k': int(OPTIMAL_K),
            'silhouette': float(silhouette_score(X_scaled, kmeans.labels_)),
            'davies_bouldin': float(davies_bouldin_score(X_scaled, kmeans.labels_)),
            'kmeans_gmm_ari': float(kmeans_gmm_ari),
            'bootstrap_stability_ari': float(np.mean(boot_aris)),
        }
    },
    'clusters': {
        str(cid): {
            'name': cluster_names[cid],
            'size': int((madonna_cleaned['cluster'] == cid).sum()),
            'profile': {f: float(cluster_means.loc[cid, f]) for f in CLUSTER_FEATURES}
        } for cid in range(OPTIMAL_K)
    },
    'diva_comparison': comparison_df.to_dict(orient='records')
}
with open('exports/web_bundle.json', 'w') as f:
    json.dump(web_bundle, f, indent=2)

print("Exported to ./exports/:")
for fname in sorted(os.listdir('exports')):
    print(f"  - exports/{fname}")


## Summary of what changed vs. v1

| Area | v1 | v2 |
|---|---|---|
| Bugs | `madonna_id` undefined crash | fixed, added schema assertions |
| Features | 5 hand-picked | 9, justified by correlation check |
| Scaling | StandardScaler | RobustScaler (audio features have outlier tails) |
| K selection | elbow only, arrow points at K=4 | elbow + silhouette + GMM BIC/AIC, must agree |
| Validation | none | KMeans-vs-GMM ARI + bootstrap stability ARI |
| Persona names | hardcoded strings | auto-derived from cluster z-scores |
| Dim. reduction | t-SNE | UMAP (better for small-N global structure) |
| Diva comparison | written 3x with drifting artist lists | one consolidated pass |
| Exports | ~15 overlapping files | 4 CSV/JSON + 1 web bundle |
